# 🔬 Notebook 02: Recurrent Deep Learning (LSTM) & Additive Forecaster Benchmarking
### Multi-Horizon (24h to 72h) Time-Series Atmospheric Modeling
**Author:** Khushi Yadav ([@khushiyadav35](https://github.com/khushiyadav35))  
**Degree:** B.Tech Computer Science & Engineering | Capstone Research Project  

---
## 1. Sequence Modeling Methodology
Forecasting atmospheric air pollution requires capturing both:
- **Short-term memory dynamics:** Lagged particulates ($t-1, t-2, t-24$) and rolling moving averages.
- **Nonlinear meteorological interactions:** Low wind speed (< 1.5 m/s) trapping inversion particulates.
- **Multi-step projection:** Generating continuous 24h to 72h ahead vectors without autoregressive error explosion.

We benchmark two contrasting paradigms:
1. **Deep PyTorch LSTM:** Multi-layer recurrent encoder with Huber Smooth-L1 Loss and Dropout.
2. **Additive Prophet Forecaster:** Fourier decomposition into diurnal, weekly, and annual harmonics with exogenous weather regressors.

In [1]:
import sys
from pathlib import Path

BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.models.lstm_forecaster import LSTMTrainer
from src.models.prophet_forecaster import AdditiveProphetForecaster
from src.preprocessing.feature_engineering import TimeSeriesFeatureEngineer
from src.data.ingestion import load_dataset_for_city
from src.preprocessing.cpcb_calculator import CPCBCalculator
from config.settings import MODELS_DIR

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 2. Inspecting Saved Benchmark Metrics

In [2]:
with open(MODELS_DIR / 'benchmark_metrics.json', 'r') as f:
    benchmarks = json.load(f)

print(json.dumps(benchmarks, indent=2))

## 3. PyTorch LSTM Inference & 95% Confidence Interval Trajectory
Evaluating the multi-step 24-hour ahead horizon on unseen test observations.

In [3]:
fe = TimeSeriesFeatureEngineer.load(MODELS_DIR / 'feature_engineer_delhi.pkl')
lstm = LSTMTrainer.load_checkpoint(MODELS_DIR / 'lstm_delhi_24h.pt')

raw_delhi = load_dataset_for_city('Delhi')
delhi_df = CPCBCalculator.compute_dataframe_naqi(raw_delhi)
feat_df = fe.engineer_features(delhi_df)

X_all, y_all = fe.get_feature_matrix(feat_df)
X_scaled, _ = fe.transform(X_all)

# Take a slice from the test set
sample_seq = X_scaled[-120:-48].reshape(1, 72, -1)
actual_future = delhi_df['cpcb_aqi'].values[-48:-24]

y_hat, y_low, y_up = lstm.predict_with_intervals(sample_seq, fe)
y_hat = y_hat[0]
y_low = y_low[0]
y_up = y_up[0]

plt.figure(figsize=(12, 5))
hours_ahead = np.arange(1, 25)
plt.plot(hours_ahead, actual_future, 'ko-', label='Ground Truth Observations (CPCB)', linewidth=2)
plt.plot(hours_ahead, y_hat, 'b*--', label='PyTorch Deep LSTM Predictions', linewidth=2.5)
plt.fill_between(hours_ahead, y_low, y_up, color='blue', alpha=0.18, label='95% Confidence Interval')

plt.title('24-Hour Multi-Step Ahead Air Quality Forecast (Delhi)', fontweight='bold')
plt.xlabel('Forecast Horizon (Hours Ahead)')
plt.ylabel('CPCB NAQI Index')
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

## 4. Conclusion & Operational Findings
- The PyTorch Deep LSTM successfully models short-term diurnal morning/evening peaks with lower MAE.
- Additive Prophet forecasters provide stable seasonal trend decomposition and interpretable harmonic coefficients.
- High Spike Recall (>200 NAQI) ensures that impending winter smog emergencies trigger timely public health advisories.